# 06 Robustness comparison

**Purpose.** Apply the paper's robustness protocol to the classical twin: four perturbations in the **original feature space** (packet loss, timing jitter, missing features, protocol-mix shift, applied before scaling and PCA), white-box FGSM and PGD in the projected space, and a gradient-free attack restricted to attacker-controllable features.

**Scope.** These results apply only to the perturbation models and attacks that were run, against single trained models. The quantum figures are those of the archived seed-42 model reported in the paper (cluster notebook 07).
**Optional.** `RERUN = True` repeats the twin evaluation for the five seeds (about 10 minutes).

In [1]:
# --- setup: work from the laptop/ folder so that relative paths resolve ---
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd
pd.set_option("display.precision", 4)
print("working folder:", ROOT)

working folder: <repo>\local_analysis


In [2]:
RERUN = False
import subprocess
if RERUN:
    subprocess.run([sys.executable, "robustness_twin_allseeds.py"])
A = json.load(open("results/twin_robustness_allseeds.json"))
def stat(kind, **kw):
    v = []
    for sd, r in A.items():
        if kind == "raw": v.append(next(x["accuracy"] for x in r["raw_space"] if x["perturbation"] == kw["p"] and x["severity"] == kw["s"]))
        elif kind == "adv": v.append(next(x["accuracy"] for x in r["adversarial_pca"] if x["attack"] == kw["a"] and x["epsilon"] == kw["e"]))
        else: v.append(next(x["accuracy"] for x in r["constrained"] if x["eps_rel"] == kw["e"]))
    v = np.array(v); return v.mean(), v.std(ddof=1)
# quantum values: archived seed-42 model, as reported in the paper (cluster notebook 07)
Q = {"jitter 20%": 0.5841, "missing 20%": 0.6733, "packet loss 20%": 0.7599, "PGD-10 eps=0.02": 0.5127, "constrained 2%": 0.5870}
Q0 = {"jitter 20%": 0.7919, "missing 20%": 0.7919, "packet loss 20%": 0.7919, "PGD-10 eps=0.02": 0.7919, "constrained 2%": 0.7880}
specs = {"jitter 20%": ("raw", dict(p="jitter", s=0.2), dict(p="jitter", s=0.0)),
         "missing 20%": ("raw", dict(p="missing", s=0.2), dict(p="missing", s=0.0)),
         "packet loss 20%": ("raw", dict(p="packet_loss", s=0.2), dict(p="packet_loss", s=0.0)),
         "PGD-10 eps=0.02": ("adv", dict(a="PGD-10", e=0.02), dict(a="PGD-10", e=0.0)),
         "constrained 2%": ("con", dict(e=0.02), dict(e=0.0))}
rows = []
for k, (kind, kw, kw0) in specs.items():
    m1, s1 = stat(kind, **kw); m0, _ = stat(kind, **kw0)
    rows.append({"condition": k, "quantum acc": Q[k], "quantum change (pp)": 100 * (Q[k] - Q0[k]),
                 "twin acc (mean of 5 seeds)": m1, "twin SD": s1, "twin change (pp)": 100 * (m1 - m0)})
pd.DataFrame(rows)

,condition,quantum acc,quantum change (pp),twin acc (mean of 5 seeds),twin SD,twin change (pp)
0,jitter 20%,0.5841,-20.78,0.6440,0.0350,-13.7467
1,missing 20%,0.6733,-11.86,0.6958,0.0209,-8.5733
2,packet loss 20%,0.7599,-3.20,0.7507,0.0065,-3.0747
3,PGD-10 eps=0.02,0.5127,-27.92,0.6099,0.1135,-17.1600
4,constrained 2%,0.5870,-20.10,0.7298,0.0138,-4.3600
